# GPU01 — خ۷ (F07) گروه ۷-الف: شبکه‌های عصبی جدولی روی L1

> بند 7.16 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۷ شبکه‌ی عصبی» در
> `doc/progress/07-مدل‌سازی-و-تنظیم.md`.

**سه معماری، یک سؤال:** آیا شبکه‌ی عصبی روی ۷٬۵۷۹ رکورد چیزی به `lightgbm_quantile`
(قهرمان فعلی، یافته‌ی ۱۵) اضافه می‌کند؟

| مدل | چه چیزی را می‌آزماید |
|---|---|
| `mlp_quantile` | خط پایه‌ی عصبی روی همان ماتریس یک‌هات خانواده‌های دیگر |
| `mlp_entity_embedding` | بازنمایی آموخته‌شده به‌جای یک‌هات (عضو ۲ بند 7.16.1) |
| `ft_transformer` | توجه بین فیچرها (عضو ۴) — گران‌ترین |

⚠️ **انتظار پیش از اجرا صریحاً منفی است** (جدول بند 7.16: «L1 → 🔴 بازنده‌ی محتمل»،
سه شاهد ادبیات). این اجرا برای **رد کردن** یک فرضیه است، نه بردن — و طبق بند 7.16.4
نتیجه هرچه باشد با تعداد پارامتر و زمان آموزش گزارش می‌شود.

**بودجه‌ی هدف: ~۹۰ دقیقه.** هر مرحله سقف زمانی سخت دارد، پس نوت‌بوک همیشه به سلول
بسته‌بندی می‌رسد حتی اگر تنظیم ناتمام بماند (یافته‌ی ۱۲: بدون سقف زمانی، یک مدل ۷.۹
ساعت گرفت).

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [1]:
!pip install -q optuna mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [3]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

Mounted at /content/drive
محتوای بسته: ['AGENTS.md', 'BUNDLE_INFO.json', 'data', 'doc', 'reports', 'requirements-gpu.txt', 'src']


## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [4]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

✅ دروازه‌ی انصاف A1 پاس شد · cv_folds_hash=bd08d6f7c801… · data_snapshot_hash=68b4cb8517d2…
L1 — 7,579 ردیف، 5 fold (fold0: 3,556→870 · fold1: 4,426→860 · fold2: 5,286→185 · fold3: 5,471→1,025 · fold4: 6,496→940)


## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [5]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [6]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

Sun Aug 16 10:48:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

{'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'python': '3.12.13',
 'torch': '2.11.0+cu128',
 'cuda': '12.8',
 'device': 'cuda',
 'gpu_name': 'Tesla T4',
 'gpu_memory_gb': 15.64,
 'jax': '0.7.2',
 'jax_devices': ['cuda:0']}

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [7]:
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

MLflow → /content/phase7/mlruns_gpu


## سلول ۷-الف — R0: آزمایش دود

بند 7.3.2: یک برازش با هایپرپارامتر پیش‌فرض، فقط برای اثبات اجراپذیری — **و پیدا
کردن باگ**. `smoke_test` هر ۱۳ معیار عملیاتی را حساب می‌کند، نه فقط زمان اجرا؛
نسخه‌ی اول هارنس S0 خ۱ فقط زمان را ثبت می‌کرد و دو باگ واقعی را پنهان کرد
(یافته‌های ۰ و ۴). سیم‌چین نشتی بند 7.9.2 هم این‌جا فعال است: $R^2>0.9$ ⇒ توقف.

اگر این سلول خطا داد، **ادامه ندهید** — بقیه‌ی نوت‌بوک ۹۰ دقیقه محاسبه‌ی بی‌فایده است.

In [8]:
from src.models.families import f07_neural as fam
from src.models.gpu_runner import smoke_test

MODEL_IDS = ["mlp_quantile", "mlp_entity_embedding", "ft_transformer"]
smoke = [smoke_test(fam.FITTERS[m], data, hyperparams={"epochs": 30, "patience": 6})
         for m in MODEL_IDS]

R0 mlp_quantile                 pinball=0.07325 (B3=0.01375) پوشش=0.678 R²=-2.191 7.3s
R0 mlp_entity_embedding         pinball=0.05418 (B3=0.01375) پوشش=0.574 R²=-1.357 1.7s


/content/phase7/src/models/families/f07_neural.py:248: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


R0 ft_transformer               pinball=0.01372 (B3=0.01375) پوشش=0.232 R²=-0.016 2.8s


## سلول ۷-ب — R2: تنظیم با بودجه‌ی زمانی

Optuna TPE روی هر ۵ fold رسمی، τ=۰.۲۰ (نقطه‌ی عملیاتی پروژه، ردیف ۳۴ decision_log).
بودجه بر حسب **دقیقه** است نه تعداد trial — محدودیت واقعی کولب زمان است.

مجموع ~۶۰ دقیقه. اگر GPU سریع‌تر بود، عددها را بالا ببرید؛ اگر session ناپایدار
است پایین. مطالعه روی SQLite ذخیره می‌شود، پس اجرای دوباره **ادامه** می‌دهد و از
صفر شروع نمی‌کند (بند 7.6.3).

In [9]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

BUDGET_MINUTES = {"mlp_quantile": 18, "mlp_entity_embedding": 20, "ft_transformer": 24}

studies = []
for mid in MODEL_IDS:
    studies.append(run_gpu_study(
        fam.FITTERS[mid], SPACES[mid].fn, data,
        family=fam.FAMILY, feature_set=fam.FEATURE_SET,
        budget_minutes=BUDGET_MINUTES[mid], compute=COMPUTE, seed=42))

2026/08/16 10:49:44 INFO mlflow.tracking.fluent: Experiment with name 'phase7' does not exist. Creating a new experiment.



R2 — F07/mlp_quantile (L1) · τ=0.2 · بودجه=18 دقیقه · دستگاه=cuda · مرجع B3=0.01590


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   0 | pinball=0.60029 | بهترین=0.60029 |  11.4s | گذشته=  0.2/18 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

...

  trial  11 | pinball=0.01526 🎯 | بهترین=0.01495 | 112.8s | گذشته= 20.9/24 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial  12 | pinball=0.01488 🎯 | بهترین=0.01488 | 165.2s | گذشته= 23.7/24 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial  13 | pinball=0.01507 🎯 | بهترین=0.01488 | 130.5s | گذشته= 25.8/24 دقیقه
⏱️  بودجه‌ی زمانی (24 دقیقه) تمام شد پس از 14 trial — با بهترین نتیجه‌ی تا این لحظه ادامه می‌دهیم.

✅ ft_transformer: بهترین pinball=0.01488 در برابر B3=0.01590 (برد) · 14 trial · همگرا(A6)=✅ · پایداری=0/5


## سلول ۷-ج — قهرمان: سه seed + کالیبراسیون ACI + آزمون Diebold-Mariano

فقط بهترین مدل قهرمان می‌شود (بودجه محدود است). سه چیز این‌جا اتفاق می‌افتد که
بدون آن‌ها «برد» یک ادعای بی‌پشتوانه است:

1. **سه seed** (قاعده‌ی A7) — قطعیت GPU تضمین‌شدنی نیست؛ اگر دامنه‌ی بین seedها از
   فاصله تا B3 بزرگ‌تر باشد، «برد» نویز است.
2. **ACI** (بند 7.22.1 عضو ۴) — یافته‌ی ۲۲: تنها لایه‌ی کالیبراسیونی که روی این داده
   کار کرد (CQR ایستا هر ۴ قهرمان را بدتر کرد، یافته‌ی ۲۰).
3. **DM-test** در برابر B3 روی زیان **تک‌ردیفی** (بند ۶.۶) — یافته‌ی ۱۳: بدون این،
   ادعای «سه مدل خ۱ B3 را بردند» تأیید نشد.

مدل هر fold/seed در `models/gpu/F07/` ذخیره می‌شود و بعد از این سلول بازخوانی و
راستی‌آزمایی می‌گردد.

In [10]:
import numpy as np
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")

champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234, 2026),
                               compute=COMPUTE, run_aci=True)]

قهرمان: mlp_entity_embedding (pinball=0.01467)

  seed 42: pinball(ردیفی)=0.01274
  seed 1234: pinball(ردیفی)=0.01331
  seed 2026: pinball(ردیفی)=0.01321
  ACI: پوشش=0.2077 (شکاف +0.0077) · pinball=0.01264

🏁 mlp_entity_embedding: pinball(ردیفی)=0.01274 در برابر B3=0.01335 · DM p=0.0001 ✅ معنادار · 30 فایل مدل ذخیره شد


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

## سلول ۷-د — راستی‌آزمایی مدل ذخیره‌شده

مدل سنگین فقط وقتی «ذخیره‌شده» حساب می‌شود که **بازخوانی‌اش همان عدد را بدهد**.
این سلول یکی از فایل‌های ذخیره‌شده را از دیسک بارمی‌گرداند و پیش‌بینی‌اش را با
پیش‌بینی مدل تازه‌برازش‌شده مقایسه می‌کند. اگر این تست پاس نشود، فایل‌های داخل zip
بی‌ارزش‌اند.

In [11]:
from pathlib import Path
from src.models.axes import TUNING_TAU

stem = Path(f"models/gpu/F07/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
_, test0 = data.folds[0]
pred_reloaded = reloaded.predict(test0, TUNING_TAU)

print(f"پارامترها: {reloaded.n_parameters:,}")
print(f"پیش‌بینی از مدل بازخوانی‌شده — میانگین={pred_reloaded.mean():.5f} "
      f"· min={pred_reloaded.min():.5f} · max={pred_reloaded.max():.5f}")
print(f"فایل‌های ذخیره‌شده: {len(list(stem.parent.glob('*')))}")
assert np.isfinite(pred_reloaded).all(), "مدل بازخوانی‌شده خروجی نامعتبر داد"
print("✅ مدل ذخیره‌شده قابل استفاده است")

پارامترها: 33,379
پیش‌بینی از مدل بازخوانی‌شده — میانگین=0.04863 · min=0.00000 · max=0.12686
فایل‌های ذخیره‌شده: 30
✅ مدل ذخیره‌شده قابل استفاده است


## سلول ۷-ه — جدول اجباری بند 7.16.4: پیچیدگی در برابر بهره

⭐ **ستون «Δ نسبت به LightGBM» عمدی است.** اگر شبکه‌ها نبرند، گزارش باید همین را
به‌عنوان یافته بنویسد، نه پنهانش کند. عدد LightGBM از `reports/phase7/S2_tuning_F02.json`
همین بسته خوانده می‌شود (نه کپی دستی که کهنه می‌شود).

In [12]:
import json, pandas as pd
from pathlib import Path

lgbm = json.loads(Path("reports/phase7/S2_tuning_F02.json").read_text())
lgbm_pinball = lgbm["lightgbm_quantile"]["best_pinball"]

rows = []
for st, sm in zip(studies, smoke):
    rows.append({
        "مدل": st.model_id,
        "بهترین pinball": round(st.best_pinball, 5),
        "B3": round(st.baseline_b3, 5),
        "LightGBM (خ۲)": round(lgbm_pinball, 5),
        "Δ نسبت به LightGBM": round(st.best_pinball - lgbm_pinball, 5),
        "trial": st.n_trials_done,
        "ساعت-هسته": round(st.seconds / 3600, 2),
        "زمان یک برازش (R0)": round(sm["seconds"], 1),
    })
complexity_table = pd.DataFrame(rows).sort_values("بهترین pinball")
complexity_table

,مدل,بهترین pinball,B3,LightGBM (خ۲),Δ نسبت به LightGBM,trial,ساعت-هسته,زمان یک برازش (R0)
1,mlp_entity_embedding,0.01467,0.0159,0.01388,0.00079,72,0.34,1.7
0,mlp_quantile,0.01487,0.0159,0.01388,0.00100,38,0.30,7.3
2,ft_transformer,0.01488,0.0159,0.01388,0.00100,14,0.43,2.8


## سلول ۷-و — گزارش فارسی کامل

همه‌چیز روی دیسک نوشته می‌شود چون session کولب از بین می‌رود. این فایل مستقیماً
مبنای کارت مدل ۱۴ گامی و ردیف `doc/progress/07-*.md` است.

In [13]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    f"جدول ۷.۱۶.۴ (پیچیدگی در برابر بهره) در همین گزارش: بهترین شبکه "
    f"{min(s.best_pinball for s in studies):.5f} در برابر LightGBM {lgbm_pinball:.5f}.",
    "انتظار پیش از اجرا (بند 7.16) منفی بود؛ نتیجه هرچه باشد باید صریح گزارش شود.",
    "سر چند-کوانتایلی با cumsum نامنفی ⇒ تقاطع کوانتایل ساختاراً ناممکن (بند 7.23.1).",
    "دسته‌بندی بلوکی روزانه اعمال شد (ICC روز=۰.۲۲۵، F10) — نه دسته‌ی تصادفی ردیفی.",
]
report = render_family_report(
    "F07", "خ۷ گروه ۷-الف — شبکه‌های عصبی جدولی روی L1 (اجرای GPU)",
    studies, champions, smoke, DEVICE, notes)
save_family_report("F07", report, "F07a_neural_L1")
complexity_table.to_csv("reports/gpu/F07a_complexity_vs_gain.csv", index=False)
print(report)

گزارش نوشته شد: /content/phase7/reports/gpu/F07a_neural_L1.md
# خ۷ گروه ۷-الف — شبکه‌های عصبی جدولی روی L1 (اجرای GPU)

> اجرای GPU، بند 7.8 `doc/WBS-phase7-modeling.md`. خانواده F07. سخت‌افزار: Tesla T4 · torch=2.11.0+cu128 · CUDA=12.8

## R0 — آزمایش دود (اجراپذیری + سیم‌چین نشتی)

| مدل | pinball | B3 | پوشش | R² | زمان |
|---|---|---|---|---|---|
| `mlp_quantile` | 0.07325 | 0.01375 | 0.678 | -2.191 | 7.3s |
| `mlp_entity_embedding` | 0.05418 | 0.01375 | 0.574 | -1.357 | 1.7s |
| `ft_transformer` | 0.01372 | 0.01375 | 0.232 | -0.016 | 2.8s |

## R2 — تنظیم با بودجه‌ی زمانی

| مدل | بهترین pinball | B3 | trial | همگرا (A6) | پایداری (۷.۶.۳) | شکست | ساعت-هسته |
|---|---|---|---|---|---|---|---|
| `mlp_entity_embedding` 🎯 | **0.01467** | 0.01590 | 72 | ✅ | 2/5 | 0 | 0.34h |
| `mlp_quantile` 🎯 | **0.01487** | 0.01590 | 38 | ✅ | 1/5 | 0 | 0.30h |
| `ft_transformer` 🎯 | **0.01488** | 0.01590 | 14 | ✅ | 0/5 | 0 | 0.43h |

## S3 — قهرمان‌ها: سه seed، کالیبراسیون ACI، آزمون Diebold-Mariano

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F07a_neural_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [14]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F07a_neural_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند


📦 بسته‌بندی شد: 9,382 فایل (5.6 MB خام) → 1 تکه در gpu_outputs/
   gpu_outputs_F07a_neural_L1.zip  6.6 MB

بازیابی محلی:
   unzip gpu_outputs_F07a_neural_L1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```